# Table 4, with seed variance

The published Table 4 reports **one stochastic run** per (policy, bot volume) cell.
That matters: the simulation resamples the population, the bot strengths, the human
abandonment draws and (for Thompson Sampling) the posterior, so a single run can sit
well off the centre of its own distribution.

This notebook repeats every cell over many seeds and reports mean +/- standard
deviation, then recomputes the four composite metrics from those repeats.

**Two different spreads appear below -- do not confuse them:**

| Spread | Meaning |
|---|---|
| `+/-` inside a simulation-type row | across **seeds**, at fixed bot volume. New here. |
| `+/-` in the **Average** block | across the **six simulation types**. This is what the paper's Table 4 reports. |

Nothing in `rlcaptcha` is modified; this notebook only calls it.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rlcaptcha import *
from rlcaptcha.simulate import SimResult

OUT = Path('results_seeded')
OUT.mkdir(exist_ok=True)

N_SEEDS = 50          # repeats per (policy, bot volume)
SEED_BASE = 1000

humans, bots = load_sessions()
print(f'{len(humans)} human sessions, {len(bots)} bot sessions')

In [ ]:
scorer = BotScorer()
cache = ScoreCache(scorer).precompute(humans, bots)

## 1. Run every cell over N seeds

Each policy keeps the reward configuration its published run used -- see
README 'Known inconsistencies'. Seeds are shared across policies so every policy
faces the same sequence of populations.

In [ ]:
POLICIES = [
    (LinUCBPolicy(),          BANDIT_REWARDS),
    (ThompsonPolicy(),        BANDIT_REWARDS),
    (DQNPolicy(),             DQN_REWARDS),
    (DQNAblationPolicy(),     DQN_REWARDS),
    (SingleThresholdPolicy(), DQN_REWARDS),
    (MultiThresholdPolicy(),  DQN_REWARDS),
]
ORDER = [p.name for p, _ in POLICIES]
METRICS = ['DI', 'BOS', 'SP-F1', 'SI-F1']

runs, trajectories = [], {}

for policy, rewards in POLICIES:
    for n_bots in EVAL_BOT_COUNTS:
        traj = []
        for k in range(N_SEEDS):
            res = run_simulation(policy, humans, bots, n_bots, cache=cache,
                                 rewards=rewards, seed=SEED_BASE + k)
            m = evaluate(res)
            runs.append({
                'Policy': policy.name, 'Bots': n_bots, 'seed': SEED_BASE + k,
                'Remaining Humans': res.surviving_humans,
                'Remaining Bots': res.surviving_bots,
                'DI': m.DI, 'BOS': m.BOS, 'SP-F1': m.SP_F1, 'SI-F1': m.SI_F1,
                'human_step_reward': res.human_step_reward,
                'bot_step_reward': res.bot_step_reward,
                'human_session_reward': res.human_session_reward,
                'bot_session_reward': res.bot_session_reward,
            })
            traj.append((res.human_counts, res.bot_counts))
        trajectories[(policy.name, n_bots)] = traj
    print(f'done: {policy.name}')

raw = pd.DataFrame(runs)
print(raw.shape)
raw.head()

## 2. Mean and standard deviation across seeds

In [ ]:
VALUE_COLS = ['Remaining Humans', 'Remaining Bots'] + METRICS

grp = raw.groupby(['Policy', 'Bots'])[VALUE_COLS]
mean_df, std_df = grp.mean(), grp.std(ddof=1)

mean_df.round(3).head(12)

## 3. Metrics two ways

The composite metrics are non-linear in the survival rates, so the mean of the metric
is not the metric of the mean. Both are reported:

* **per-seed** -- evaluate each run, then average. This is the honest estimate, and it
  is what carries a meaningful standard deviation.
* **of-means** -- build one synthetic result from the mean survivor counts and evaluate
  that once. This is the direct 'compute the metrics from the mean' reading, and it has
  no spread of its own.

Where the two disagree, the gap is the non-linearity, and it is largest for the
high-variance policies.

In [ ]:
rows = []
for (name, n_bots), row in mean_df.iterrows():
    synthetic = SimResult(
        n_humans=EVAL_HUMANS, n_bots=n_bots,
        surviving_humans=row['Remaining Humans'],
        surviving_bots=row['Remaining Bots'],
        human_step_reward=0.0, bot_step_reward=0.0,
        human_session_reward=0.0, bot_session_reward=0.0,
    )
    m = evaluate(synthetic)
    rows.append({'Policy': name, 'Bots': n_bots,
                 'DI': m.DI, 'BOS': m.BOS, 'SP-F1': m.SP_F1, 'SI-F1': m.SI_F1})

of_means = pd.DataFrame(rows).set_index(['Policy', 'Bots'])

comparison = pd.concat(
    {'per-seed mean': mean_df[METRICS], 'of-means': of_means,
     'difference': of_means - mean_df[METRICS]}, axis=1)
comparison.round(3)

## 4. Table 4, paper format

Rows grouped by simulation type exactly as in the manuscript. Values are
mean +/- std **across seeds**. The Average block at the bottom keeps the paper's own
convention: mean +/- std **across the six simulation types**, computed from the
per-simulation means.

In [ ]:
def cell(mu, sd, metric):
    if metric in ('Remaining Humans', 'Remaining Bots'):
        return f'{mu:.1f} +/- {sd:.1f}'
    if metric == 'DI':
        return f'{mu:.1f} +/- {sd:.1f}'
    return f'{mu:.3f} +/- {sd:.3f}'

ROW_ORDER = ['Remaining Humans', 'Remaining Bots'] + METRICS
LOWER_IS_BETTER = {'Remaining Bots'}

records, best_mask = [], []
for n_bots in EVAL_BOT_COUNTS:
    for metric in ROW_ORDER:
        mus = [mean_df.loc[(p, n_bots), metric] for p in ORDER]
        sds = [std_df.loc[(p, n_bots), metric] for p in ORDER]
        rec = {'Simulation Type': f'{n_bots} Bots', 'Metric': metric}
        for p, mu, sd in zip(ORDER, mus, sds):
            rec[p] = cell(mu, sd, metric)
        records.append(rec)
        target = min(mus) if metric in LOWER_IS_BETTER else max(mus)
        best_mask.append([abs(v - target) < 1e-9 for v in mus])

# Average block: across the six simulation types, as the paper does.
for metric in METRICS:
    mus, sds = [], []
    for p in ORDER:
        per_sim = [mean_df.loc[(p, b), metric] for b in EVAL_BOT_COUNTS]
        mus.append(float(np.mean(per_sim)))
        sds.append(float(np.std(per_sim, ddof=0)))
    rec = {'Simulation Type': 'Average', 'Metric': metric}
    for p, mu, sd in zip(ORDER, mus, sds):
        rec[p] = cell(mu, sd, metric)
    records.append(rec)
    best_mask.append([abs(v - max(mus)) < 1e-9 for v in mus])

table4 = pd.DataFrame(records)[['Simulation Type', 'Metric'] + ORDER]
table4

## 5. How the published single run sits inside its own distribution

For each published value, the z-score against the seeded distribution. A magnitude
above ~2 means the single run the paper reports was an unusual draw.

In [ ]:
PAPER = {
    'LinUCB':                  {0: (100, 0), 20: (96, 2), 100: (66, 13), 200: (35, 3), 500: (29, 7), 1000: (28, 15)},
    'Thompson Sampling':       {0: (100, 0), 20: (100, 1), 100: (98, 11), 200: (22, 13), 500: (28, 31), 1000: (41, 41)},
    'DQN':                     {0: (89, 0), 20: (94, 4), 100: (79, 16), 200: (81, 19), 500: (78, 29), 1000: (75, 22)},
    'DQN without H-Score':     {0: (98, 0), 20: (98, 13), 100: (97, 72), 200: (96, 116), 500: (98, 347), 1000: (96, 661)},
    'Static Single-Threshold': {0: (51, 0), 20: (58, 1), 100: (49, 8), 200: (53, 15), 500: (51, 44), 1000: (52, 98)},
    'Static Multi-Threshold':  {0: (69, 0), 20: (65, 1), 100: (69, 15), 200: (70, 41), 500: (68, 79), 1000: (65, 170)},
}

z_rows = []
for p in ORDER:
    for n_bots in EVAL_BOT_COUNTS:
        ph, pb = PAPER[p][n_bots]
        mh = mean_df.loc[(p, n_bots), 'Remaining Humans']
        sh = std_df.loc[(p, n_bots), 'Remaining Humans']
        mb = mean_df.loc[(p, n_bots), 'Remaining Bots']
        sb = std_df.loc[(p, n_bots), 'Remaining Bots']
        z_rows.append({
            'Policy': p, 'Bots': n_bots,
            'paper humans': ph, 'seeded humans': round(mh, 1),
            'sd humans': round(sh, 2),
            'z humans': round((ph - mh) / sh, 2) if sh > 0 else np.nan,
            'paper bots': pb, 'seeded bots': round(mb, 1),
            'sd bots': round(sb, 2),
            'z bots': round((pb - mb) / sb, 2) if sb > 0 else np.nan,
        })

zscores = pd.DataFrame(z_rows)
outliers = zscores[(zscores['z humans'].abs() > 2) | (zscores['z bots'].abs() > 2)]
print(f'{len(outliers)} of {len(zscores)} published cells lie beyond 2 sd of their own distribution')
zscores

## 6. Save results

In [ ]:
with pd.ExcelWriter(OUT / 'RL_Captcha_Results_Seeded.xlsx', engine='openpyxl') as xl:
    table4.to_excel(xl, sheet_name='Table4 (paper format)', index=False)
    mean_df.round(4).to_excel(xl, sheet_name='Mean across seeds')
    std_df.round(4).to_excel(xl, sheet_name='Std across seeds')
    of_means.round(4).to_excel(xl, sheet_name='Metrics of mean counts')
    comparison.round(4).to_excel(xl, sheet_name='Per-seed vs of-means')
    zscores.to_excel(xl, sheet_name='Published run z-scores', index=False)
    raw.round(4).to_excel(xl, sheet_name='Raw runs', index=False)

raw.to_csv(OUT / 'raw_runs.csv', index=False)
meta = {'n_seeds': N_SEEDS, 'seed_base': SEED_BASE,
        'humans_per_sim': EVAL_HUMANS, 'bot_counts': list(EVAL_BOT_COUNTS),
        'policies': {p.name: ('BANDIT_REWARDS' if r is BANDIT_REWARDS else 'DQN_REWARDS')
                     for p, r in POLICIES}}
(OUT / 'run_config.json').write_text(json.dumps(meta, indent=2))
print('written to', OUT.resolve())

### Table 4 as a PDF, paper layout

Best value per row in bold, matching the manuscript's convention.

In [ ]:
n_rows = len(table4)
fig, ax = plt.subplots(figsize=(17, 0.232 * n_rows + 1.5))
ax.axis('off')

tbl = ax.table(cellText=table4.values, colLabels=table4.columns,
               cellLoc='center', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(7.5)
# Pin row heights so the table fills the axes exactly instead of overflowing it.
for c in tbl.get_celld().values():
    c.set_height(1.0 / (n_rows + 1))

n_cols = len(table4.columns)
for c in range(n_cols):
    h = tbl[0, c]
    h.set_facecolor('#20304a')
    h.set_text_props(color='white', fontweight='bold')

def band_colour(sim):
    if sim == 'Average':
        return '#dfe6f0'
    idx = list(EVAL_BOT_COUNTS).index(int(sim.split()[0]))
    return '#eef1f6' if idx % 2 == 0 else 'white'

for r in range(n_rows):
    sim = table4.iloc[r]['Simulation Type']
    band = band_colour(sim)
    for c in range(n_cols):
        cl = tbl[r + 1, c]
        cl.set_facecolor(band)
        if c >= 2 and best_mask[r][c - 2]:
            cl.set_text_props(fontweight='bold')
        if c < 2:
            cl.set_text_props(fontweight='bold' if sim == 'Average' else 'normal')

fig.suptitle('Performance Comparison of CAPTCHA Orchestration Systems across Different '
             'Bot Attack Scenarios (Fixed 100 Humans)\n'
             f'mean +/- sd over {N_SEEDS} seeds; Average block is mean +/- sd across the six '
             'simulation types',
             fontsize=11, y=0.985)
plt.subplots_adjust(left=0.01, right=0.99, top=0.93, bottom=0.01)
plt.savefig(OUT / 'table4_seeded.pdf', dpi=300, bbox_inches='tight')
plt.show()

### Population dynamics, averaged over seeds

Solid line is the mean across seeds; the band is +/- one standard deviation.

In [ ]:
sims = list(EVAL_BOT_COUNTS)
fig, axes = plt.subplots(len(sims), len(ORDER),
                         figsize=(3.2 * len(ORDER), 2.6 * len(sims)),
                         sharex=True, sharey=True)

for col, name in enumerate(ORDER):
    for row, n_bots in enumerate(sims):
        ax = axes[row, col]
        traj = trajectories[(name, n_bots)]
        width = max(len(h) for h, _ in traj)
        def pad(seq):
            return np.array([list(s) + [s[-1]] * (width - len(s)) for s in seq], dtype=float)
        H = pad([h for h, _ in traj])
        B = pad([b for _, b in traj])
        t = np.arange(width)
        for arr, colour, label in ((H, 'tab:blue', 'Humans'), (B, 'tab:red', 'Bots')):
            mu, sd = arr.mean(0), arr.std(0)
            ax.plot(t, mu, color=colour, lw=1.8, label=label,
                    ls='-' if label == 'Humans' else '--')
            ax.fill_between(t, mu - sd, mu + sd, color=colour, alpha=0.18)
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.3)
        if row == 0:
            ax.set_title(name, fontsize=10)
        if col == 0:
            ax.set_ylabel(str(n_bots) + ' Bots', fontsize=10)
        if row == len(sims) - 1:
            ax.set_xlabel('Timestep')

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.975])
plt.savefig(OUT / 'population_dynamics_seeded.pdf', dpi=300)
plt.show()

### Ablation study with seed error bars

Figure 4 of the manuscript, now with genuine run-to-run uncertainty rather than a
single draw per point.

In [ ]:
ABLATION = ['DQN', 'DQN without H-Score',
            'Static Single-Threshold', 'Static Multi-Threshold']
x = np.arange(len(EVAL_BOT_COUNTS))

fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
for ax, metric in zip(axes.ravel(), METRICS):
    for name in ABLATION:
        mu = np.array([mean_df.loc[(name, b), metric] for b in EVAL_BOT_COUNTS])
        sd = np.array([std_df.loc[(name, b), metric] for b in EVAL_BOT_COUNTS])
        line = ax.errorbar(x, mu, yerr=sd, marker='o', capsize=3, lw=1.6, label=name)
        ax.axhline(mu.mean(), color=line[0].get_color(), ls=':', alpha=0.55)
    ax.set_xticks(x)
    ax.set_xticklabels(EVAL_BOT_COUNTS)
    ax.set_xlabel('Number of bots')
    ax.set_title(metric)
    ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=9)
fig.suptitle('Ablation study, mean +/- sd over ' + str(N_SEEDS) + ' seeds', fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(OUT / 'ablation_study_seeded.pdf', dpi=300)
plt.show()

### All six policies, per metric

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
for ax, metric in zip(axes.ravel(), METRICS):
    for name in ORDER:
        mu = np.array([mean_df.loc[(name, b), metric] for b in EVAL_BOT_COUNTS])
        sd = np.array([std_df.loc[(name, b), metric] for b in EVAL_BOT_COUNTS])
        ax.errorbar(x, mu, yerr=sd, marker='o', capsize=3, lw=1.6, label=name)
    ax.set_xticks(x)
    ax.set_xticklabels(EVAL_BOT_COUNTS)
    ax.set_xlabel('Number of bots')
    ax.set_title(metric)
    ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=8)
fig.suptitle('All policies, mean +/- sd over ' + str(N_SEEDS) + ' seeds', fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(OUT / 'all_policies_seeded.pdf', dpi=300)
plt.show()

In [ ]:
sorted(p.name for p in OUT.iterdir())